<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Ungraded_Assignments/Ungraded_A2_PINN4Opt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Assignment 2
- Question 1: Solve the below problem where we use PINN for an optimization task

- Question 2: Write a short para for an application in your discipline where you could use PINN. NOte that PINN can be used for various tasks, we saw four examples in class: foward problem, reverse problem, downscaling, and optimization.

# Using PINN for optimization tasks
Seo, J. Solving real-world optimization tasks using physics-informed neural computing. Sci Rep 14, 202 (2024). https://doi.org/10.1038/s41598-023-49977-3
https://www.nature.com/articles/s41598-023-49977-3

Note: all NN use until now were supervised learning. Here it is unsupervised learning, using only physical system knowledge to constrain the outputs.

---


# **Assignment**: Solve for optimal real-time battery charging /discharging decisions that a battery operator can use to maximize their profit
Your assignment is a simplified version of problems typical in energy grids. (Sample realistic casestdy: Favaro et;.al, Neural network informed day-ahead scheduling of pumped hydro energy storage, Energy, 2024 10.1016/j.energy.2023.129999 )

## Background
Locational Marginal Pricing.
In grid-scale energy market, the market price P(t) is a two-way wholesale rate known as the Locational Marginal Price (LMP). Unlike a traditional retail consumer tariff, the battery owner acts as an active participant on both sides of the market grid, bound by a single universal rate:
$\text{Instantaneous Net Financial Cost}=C(t)\cdot P(t)-D(t)\cdot P(t)$
* When Charging $(C(t) > 0)$: The battery asset is an electricity consumer. The owner pays the grid the wholesale market rate P(t) per MWh consumed.
* When Discharging $(D(t) > 0)$: The battery asset is an electricity supplier. The owner gets paid (sells to) the grid the wholesale market rate $P(t)$ per MWh supplied.

Train a network to map high-resolution continuous time-steps $t$ and real 5-minute ISO-NE real-time spot market prices $P(t)$ directly to optimal control profiles while strictly respecting internal physical boundaries.

he Power of Arbitrage and Negative Pricing: The primary objective of the optimization model is to navigate this pricing curve to minimize net spending (which mathematically maximizes trading profits). In real-world ISO New England Real-Time 5-Minute Market data, we will encounter periods where the price drops below zero (P(t) < 0). This occurs during extreme renewable oversupply events (e.g., massive midday solar generation).A negative cost equals a profit. When prices drop below zero, the grid literally pays the battery asset owner to swallow excess power and charge up the reservoir. The objetcive is to exploit these high-yield pricing troughs while protecting the chemical capacity ceilings of the battery.



Mathematical Constraints:

The model should be bound by the continuous-time dynamics of a 100 MWh Lithium-Ion Battery asset over a 24-hour sequence $(t \in [0, 24])$:
1. State of Charge (SoC) Conservation ODE:
$$\frac{dSoC(t)}{dt}=\eta \cdot C(t)-\frac{1}{\eta }\cdot D(t)$$
2. Asset Boundaries & Mutual Exclusivity:
$$0.0\le SoC(t)\le 100.0\text{ MWh}$$
$$0.0\le C(t)\le 20.0\text{ MW}\quad \text{and}\quad 0.0\le D(t)\le 20.0\text{ MW}$$
$$C(t)\cdot D(t)=0$$
3. Cyclical Boundary Limits:
$$SoC(0)=50.0\text{ MWh}\quad \text{and}\quad SoC(23.9167)=50.0\text{ MWh}$$
4. Economic Market Objective: Minimizes net grid spending (maximizes arbitrage return):
$$\text{Minimize}\int _{0}^{24}\left(C(t)\cdot P(t)-D(t)\cdot P(t)\right)dt$$

#  Some pointers on setting up the model for above setup
* Note that this is unsupervized learning. Supervised learning would be using labels -say historical human decisions - which will likely not be optimal. The physical system properties help constrain the training.
* What should be the NN architecture?
  * We have $C(t).D(t)=0$, in other words, you cannot charge and discharge at the same time (only one of it can be positive, so think of combining them into one output)
* Loss function has to be differentiable - how to add loss corresponding to boundary conditions - think of ways to incorporate differentiable functions.

# Sample code to download ISO-NE data
ISO New England is an independent, nonprofit company authorized by the Federal Energy Regulatory Commission (FERC) to perform grid operations, market administration, and power systems planning. It operates in NE - Connecticut, Rhode Island, Massachusetts, Vermont, New Hampshire, and most of Maine.

In [ ]:
# =====================================================================
# STEP 1: INSTALL UNIFORM ISO DATA ACCESS COMPONENT
# =====================================================================
!pip install gridstatus -q

import pandas as pd
import numpy as np
import gridstatus

# =====================================================================
# STEP 2: EXTRACT REAL 5-MINUTE DATA FOR A SPECIFIED DATE RANGE
# =====================================================================
START_DATE = "2026-06-01"
END_DATE = "2026-06-07"

print(f"--- Querying Live ISO-NE 5-Minute API From {START_DATE} to {END_DATE} ---")
iso = gridstatus.ISONE()

# FIXED: Use the correct standardized get_lmp method with market="REAL_TIME_5_MIN"
df_raw = iso.get_lmp(
    date=START_DATE,
    end=END_DATE,
    market="REAL_TIME_5_MIN"
)

print(f"Raw 5-minute data extracted! Total records fetched: {len(df_raw)}")

# =====================================================================
# STEP 3: CONVERT 5-MINUTE TIMESTAMPS FOR TRAJECTORY PIPELINE
# =====================================================================
df_raw['Interval Start'] = pd.to_datetime(df_raw['Interval Start'])
df_raw['Date'] = df_raw['Interval Start'].dt.date

# Convert Hour to a true continuous float matching 5-minute offsets (e.g., 12:05 -> 12.083)
df_raw['Hour'] = df_raw['Interval Start'].dt.hour + (df_raw['Interval Start'].dt.minute / 60.0)

# Isolate the primary internal pricing Hub node
hub_mask = df_raw['Location'].str.contains('HUB|INTERNAL', case=False, na=False)
df_hub = df_raw[hub_mask].copy()

df_clean = df_hub[['Date', 'Hour', 'LMP']].sort_values(by=['Date', 'Hour']).reset_index(drop=True)
unique_days = df_clean['Date'].unique()

X_list = []
print(f"Structuring records into complete 288-point daily sequences...")

for day in unique_days:
    day_data = df_clean[df_clean['Date'] == day]

    # 288 unique 5-minute intervals make up one standard 24-hour day profile
    if len(day_data) == 288:
        for _, row in day_data.iterrows():
            X_list.append([row['Hour'], row['LMP']])

# Overwrite localized workspace payload
df_final = pd.DataFrame(X_list, columns=['time_hours', 'grid_price_per_mwh'])
df_final.to_csv('historical_market_prices.csv', index=False)

print(f"\n5-Minute Dataset Preparation Complete!")
print(f"Saved to: 'historical_market_prices.csv'")
print(f"Total tracking samples: {len(df_final)} coordinates across {len(df_final)//288} clean days.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.2/551.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB

100%|██████████| 6/6 [00:47<00:00,  7.87s/it]


Raw 5-minute data extracted! Total records fetched: 2086044
Structuring records into complete 288-point daily sequences...

5-Minute Dataset Preparation Complete!
Saved to: 'historical_market_prices.csv'
Total tracking samples: 1440 coordinates across 5 clean days.


# Assignment: write code for PINN
# Submit plots for C,D, P, SoC (see sample below). Also plot per day (24 hour) economic market objective.



<p align="center">
  <img src="https://raw.githubusercontent.com/diseasemodeling/MIE_525_625/main/figs/PINN4Opt.png" width="80%">
  <br>
  
</p>